# 57. 5개 민감도 설정 고정 다수결과 예측 감사
56번 결과의 최고설정을 고르지 않고5개중3개 동의라는 고정 규칙을 시험한다. 기존47앵커는 보존한다. 참조20/30/40의 일부 결과를 확인한 이후 선택한 추가 탐색이며 새 검증자료가 아니다. 5개는 독립 표본/통계적 재현5회가 아니라 같은샷에 대한 모델설정이다.


In [ ]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
p=pd.read_csv(tables/'rg3_pdf_sensitivity_predictions.csv',parse_dates=['TimeStamp'])
p=p[p.model.eq('thermal_change_low_0.05')]
assert p.configuration.nunique()==5
raw=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp']).set_index('TimeStamp')
rows=[];out=[];votes=[]
for scheme,g in p.groupby('scheme'):
 wide=g.pivot(index='TimeStamp',columns='configuration',values='prediction')
 assert wide.notna().all().all()
 truth=g.groupby('TimeStamp').actual.first().loc[wide.index].to_numpy(int)
 assert np.array_equal(truth,raw.loc[wide.index,'PassOrFail'].to_numpy(int))
 assert g.groupby('TimeStamp').actual.nunique().eq(1).all()
 assert g.groupby('TimeStamp').anchor_prediction.nunique().eq(1).all()
 anc=g.groupby('TimeStamp').anchor_prediction.first().loc[wide.index].to_numpy(bool)
 count=wide.sum(axis=1).to_numpy(int)
 pr=anc|(count>=3)
 assert ((~anc)|pr).all()
 tn,fp,fn,tp=confusion_matrix(truth,pr,labels=[0,1]).ravel()
 rows.append(dict(scheme=scheme,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn),precision=tp/(tp+fp) if tp+fp else 0,recall=tp/(tp+fn)))
 for j,t in enumerate(wide.index):
  out.append(dict(scheme=scheme,TimeStamp=t,actual=int(truth[j]),prediction=int(pr[j]),anchor_prediction=int(anc[j]),positive_votes=int(count[j])))
  if truth[j]==1:
   row=dict(scheme=scheme,TimeStamp=t,anchor_prediction=int(anc[j]),prediction=int(pr[j]),positive_votes=int(count[j]))
   row.update({k:int(wide.iloc[j][k]) for k in wide.columns});votes.append(row)
m=pd.DataFrame(rows);a=pd.DataFrame(out);v=pd.DataFrame(votes)
for obj,name in [(m,'metrics'),(a,'predictions'),(v,'all_gas_votes')]:
 obj.to_csv(tables/f'rg3_pdf_fixed_majority_{name}.csv',index=False,encoding='utf-8-sig')
print('FIXED 3 of 5 MAJORITY',m.to_string(index=False))
print('GAS AGREEMENT',v.to_string(index=False))
